In [ ]:
import os
import re
import pandas as pd
import glob
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def should_exclude_file(file_path):
    """Check if a file should be excluded from dataset statistics.
    
    Excludes JPF instrumentation and generalized files but includes EvoSuite tests.
    Based on patterns from CleanupTask.java.
    """
    file_name = os.path.basename(file_path)
    
    # Exclude JPF instrumentation files
    if file_name.startswith('_') and ('_Driver_' in file_name or '_Instrumented_' in file_name):
        return True
    
    # Exclude generalized files
    if file_name.startswith('_') and '_Generalized_' in file_name:
        return True
    
    # Include everything else (including EvoSuite tests ending with ESTest.java)
    return False

def count_line_types(file_path):
    code_lines = 0
    comment_lines = 0
    empty_lines = 0
    in_block_comment = False

    with open(file_path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                empty_lines += 1
                continue

            if in_block_comment:
                comment_lines += 1
                if '*/' in stripped:
                    in_block_comment = False
                continue

            if stripped.startswith('/*'):
                comment_lines += 1
                if not '*/' in stripped:
                    in_block_comment = True
                continue

            if stripped.startswith('//'):
                comment_lines += 1
                continue

            if '/*' in stripped:
                before_comment = stripped.split('/*', 1)[0].strip()
                if before_comment:
                    code_lines += 1
                else:
                    comment_lines += 1
                if not '*/' in stripped:
                    in_block_comment = True
                continue

            if '//' in stripped:
                before_comment = stripped.split('//', 1)[0].strip()
                if before_comment:
                    code_lines += 1
                else:
                    comment_lines += 1
                continue

            code_lines += 1

    return code_lines, comment_lines, empty_lines

def count_test_methods(content):
    test_annotation_pattern = re.compile(
        r'@(?:org\.junit(?:\.jupiter\.api)?\.)?(Test|RepeatedTest|ParameterizedTest)\b'
    )
    method_pattern = re.compile(
        r'\b\w[\w<>\[\],\s]*\s+\w+\s*\('
    )

    lines = content.splitlines()
    num_test_methods = 0
    pending_test_method = False
    method_decl = ''
    for line in lines:
        stripped = line.strip()
        # Ignore single-line comments and JavaDoc lines
        if stripped.startswith('//') or stripped.startswith('*') or stripped.startswith('/*') or stripped.startswith('*/'):
            continue

        # If annotation and method on the same line
        if test_annotation_pattern.search(stripped) and method_pattern.search(stripped):
            num_test_methods += 1
            pending_test_method = False
            method_decl = ''
            continue

        # If we see a test annotation, set the flag
        if test_annotation_pattern.search(stripped):
            pending_test_method = True
            method_decl = ''
            continue

        # If we're waiting for a method after a test annotation
        if pending_test_method:
            # Skip blank lines and annotation lines
            if not stripped or stripped.startswith('@') or stripped.startswith('//') or stripped.startswith('*') or stripped.startswith('/*') or stripped.startswith('*/'):
                continue
            # Accumulate lines for multi-line method declarations
            method_decl += ' ' + stripped
            if '(' in method_decl:
                if method_pattern.search(method_decl):
                    num_test_methods += 1
                    pending_test_method = False
                    method_decl = ''
            continue

        # Reset method_decl if not in pending state
        method_decl = ''
    return num_test_methods

def collect_stats(directory):
    num_files = 0
    num_classes = 0
    num_code_lines = 0
    num_comment_lines = 0
    num_empty_lines = 0
    num_test_methods = 0

    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".java"):
                file_path = os.path.join(root, file)
                
                # Skip generated files (but include EvoSuite tests)
                if should_exclude_file(file_path):
                    continue
                
                num_files += 1
                code, comment, empty = count_line_types(file_path)
                num_code_lines += code
                num_comment_lines += comment
                num_empty_lines += empty

                with open(file_path, encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                    num_classes += len(re.findall(r'\b(class|enum)\s+\w+', content))
                    num_test_methods += count_test_methods(content)

    return {
        "directory": directory,
        "num_files": num_files,
        "num_classes": num_classes,
        "num_code_lines": num_code_lines,
        "num_comment_lines": num_comment_lines,
        "num_empty_lines": num_empty_lines,
        "num_test_methods": num_test_methods,
    }

def get_projects_path():
    """Get projects directory path from environment variable with fallback."""
    projects_path = os.getenv('PROJECTS_PATH')
    if projects_path and Path(projects_path).exists():
        return Path(projects_path)
    
    # Fallback to relative path from notebook location
    fallback_path = Path("../../projects")
    if fallback_path.exists():
        return fallback_path.resolve()
    
    raise FileNotFoundError("Projects directory not found. Set PROJECTS_PATH in .env or ensure ../../projects exists.")

# Get projects directory dynamically
projects_root = get_projects_path()
print(f"Using projects directory: {projects_root}")

# Discover all project directories and their src/main and src/test paths
directories = []

# Non-GitHub projects (commons-utils*, eqbench*)
for pattern in ["commons-utils*", "eqbench*"]:
    for project_dir in projects_root.glob(pattern):
        if project_dir.is_dir():
            for src_type in ["main", "test"]:
                # Try both java and code subdirectories (eqbench uses code, others use java)
                for subdir in ["java", "code"]:
                    src_dir = project_dir / "src" / src_type / subdir
                    if src_dir.exists():
                        directories.append(str(src_dir))
                        break  # Only add one per src_type

# GitHub projects (github_com_*)
for project_dir in projects_root.glob("github_com_*"):
    if project_dir.is_dir():
        for src_type in ["main", "test"]:
            src_dir = project_dir / "src" / src_type / "java"
            if src_dir.exists():
                directories.append(str(src_dir))

print(f"Found {len(directories)} source directories to analyze")

# Helper to extract project name from directory path
def get_project_name(directory):
    # Assumes project is the last segment before 'src'
    parts = Path(directory).parts
    for i, part in enumerate(parts):
        if part == "src" and i > 0:
            return parts[i-1]
    return directory

# Collect statistics for all directories
stats = []
for d in directories:
    stat = collect_stats(d)
    stat["project"] = get_project_name(d)
    # Determine type by checking if path contains /main/ or /test/
    stat["type"] = "main" if "/main/" in d else "test"
    stats.append(stat)

df = pd.DataFrame(stats)
display(df)

# Aggregate by project
summary = []

# --- AGGREGATE: All non-github projects individually ---
non_github_df = df[~df["project"].str.startswith("github_com_")]
for project in non_github_df["project"].unique():
    main = non_github_df[(non_github_df["project"] == project) & (non_github_df["type"] == "main")]
    test = non_github_df[(non_github_df["project"] == project) & (non_github_df["type"] == "test")]
    summary.append({
        "project": project,
        "main_files": int(main["num_files"].sum()),
        "main_classes": int(main["num_classes"].sum()),
        "main_sloc": int(main["num_code_lines"].sum()),
        "test_files": int(test["num_files"].sum()),
        "test_classes": int(test["num_classes"].sum()),
        "test_sloc": int(test["num_code_lines"].sum()),
        "test_methods": int(test["num_test_methods"].sum()),
        "is_github": False,
    })

# --- AGGREGATE: github_com_* projects (total, mean, median) ---
github_df = df[df["project"].str.startswith("github_com_")]

if not github_df.empty:
    # Group by project and type
    github_projects = github_df["project"].unique()
    github_rows = []
    for project in github_projects:
        main = github_df[(github_df["project"] == project) & (github_df["type"] == "main")]
        test = github_df[(github_df["project"] == project) & (github_df["type"] == "test")]
        github_rows.append({
            "main_files": int(main["num_files"].sum()),
            "main_classes": int(main["num_classes"].sum()),
            "main_sloc": int(main["num_code_lines"].sum()),
            "test_files": int(test["num_files"].sum()),
            "test_classes": int(test["num_classes"].sum()),
            "test_sloc": int(test["num_code_lines"].sum()),
            "test_methods": int(test["num_test_methods"].sum()),
        })
    # Convert to DataFrame for easy stats
    github_stats = pd.DataFrame(github_rows)

    # Total
    summary.append({
        "project": "repo-reapers (total)",
        "main_files": int(github_stats["main_files"].sum()),
        "main_classes": int(github_stats["main_classes"].sum()),
        "main_sloc": int(github_stats["main_sloc"].sum()),
        "test_files": int(github_stats["test_files"].sum()),
        "test_classes": int(github_stats["test_classes"].sum()),
        "test_sloc": int(github_stats["test_sloc"].sum()),
        "test_methods": int(github_stats["test_methods"].sum()),
        "is_github": True,
    })
    # Mean
    summary.append({
        "project": "repo-reapers (mean)",
        "main_files": int(github_stats["main_files"].mean()),
        "main_classes": int(github_stats["main_classes"].mean()),
        "main_sloc": int(github_stats["main_sloc"].mean()),
        "test_files": int(github_stats["test_files"].mean()),
        "test_classes": int(github_stats["test_classes"].mean()),
        "test_sloc": int(github_stats["test_sloc"].mean()),
        "test_methods": int(github_stats["test_methods"].mean()),
        "is_github": True,
    })
    # Median
    summary.append({
        "project": "repo-reapers (median)",
        "main_files": int(github_stats["main_files"].median()),
        "main_classes": int(github_stats["main_classes"].median()),
        "main_sloc": int(github_stats["main_sloc"].median()),
        "test_files": int(github_stats["test_files"].median()),
        "test_classes": int(github_stats["test_classes"].median()),
        "test_sloc": int(github_stats["test_sloc"].median()),
        "test_methods": int(github_stats["test_methods"].median()),
        "is_github": True,
    })

df_summary = pd.DataFrame(summary)

# --- Build LaTeX table ---
def get_base_project(name):
    if name.startswith("repo-reapers"):
        return "repo-reapers"
    return name.split('-es-')[0]

def fmt(x):
    return f"{x:,}"

# Import macro mapping functions
from teralizer.exports import save_latex_table, save_csv_data, get_dataset_macro, get_table_group_order, get_project_within_type_order

def get_project_display_name(project_name):
    """Get the proper display name for a project, using LaTeX macros where available."""
    # Special handling for repo-reapers projects
    if project_name.startswith("repo-reapers"):
        # These use the DatasetRepoReapers macro
        if "total" in project_name:
            return r"\DatasetRepoReapers{} (total)"
        elif "mean" in project_name:
            return r"\DatasetRepoReapers{} (mean)"
        elif "median" in project_name:
            return r"\DatasetRepoReapers{} (median)"
        else:
            return r"\DatasetRepoReapers{}"
    
    # Try to get LaTeX macro for known datasets
    try:
        return get_dataset_macro(project_name)
    except KeyError:
        # Fallback to project name if no macro available
        return project_name

latex_table = r"""\begin{table}[H]
  \caption{Number of files, classes, source lines of code (SLOC), and test methods per project.}
  \label{tab:dataset-statistics}
  \begin{tabular}{lrrrrrrr}
    \toprule
    & \multicolumn{3}{c}{Implementation} & \multicolumn{4}{c}{Test} \\
    \cmidrule(lr){2-4} \cmidrule(lr){5-8}
    Project & Files & Classes & SLOC & Files & Classes & SLOC & Methods \\
    \midrule
"""

# Get project ordering functions (same pattern as other notebooks)
project_within_type_order = get_project_within_type_order()

# Sort all projects using two-level sorting (same pattern as other notebooks)
def get_sort_key(project_name):
    return (
        get_table_group_order(project_name, "INITIAL"),
        project_within_type_order.get(project_name, 99)
    )

df_summary['sort_key'] = df_summary['project'].apply(get_sort_key)
df_summary_sorted = df_summary.sort_values('sort_key')

# Generate table rows with proper grouping
prev_group_order = None
for _, row in df_summary_sorted.iterrows():
    current_group_order = get_table_group_order(row["project"], "INITIAL")
    
    # Add midrule between different table groups
    if prev_group_order is not None and current_group_order != prev_group_order:
        latex_table += "    \\midrule\n"
    prev_group_order = current_group_order
    
    # Use macro mapping for project name
    display_name = get_project_display_name(row["project"])
    
    latex_table += (
        f"    {display_name} & "
        f"{fmt(row['main_files'])} & {fmt(row['main_classes'])} & {fmt(row['main_sloc'])} & "
        f"{fmt(row['test_files'])} & {fmt(row['test_classes'])} & {fmt(row['test_sloc'])} & {fmt(row['test_methods'])} \\\\\n"
    )

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)

# Save the LaTeX table
save_latex_table(latex_table, 'tab-dataset-statistics')

# Export dataset statistics as CSV for LLM processing
dataset_statistics_data = []

for _, row in df_summary.iterrows():
    dataset_statistics_data.append({
        'project_name': row['project'],
        'is_github_project': bool(row['is_github']),
        'implementation_files': int(row['main_files']),
        'implementation_classes': int(row['main_classes']),
        'implementation_sloc': int(row['main_sloc']),
        'test_files': int(row['test_files']),
        'test_classes': int(row['test_classes']),
        'test_sloc': int(row['test_sloc']),
        'test_methods': int(row['test_methods'])
    })

# Create DataFrame and save CSV
dataset_statistics_df = pd.DataFrame(dataset_statistics_data)

csv_path = save_csv_data(
    dataset_statistics_df,
    'dataset-statistics-data',
    'Dataset statistics showing files, classes, SLOC, and test methods per project'
)

print(f"Dataset statistics exported to: {csv_path}")
print(f"Shape: {dataset_statistics_df.shape}")
print(f"Sample data:")
display(dataset_statistics_df.head())